# 01 — Explore Results

Interactively explore the Group 3 simulation outputs stored in `Group3.npz`.

**Prerequisites**: Stage 10 must have completed and `outputs/Group3.npz` must exist.

No FEFLOW / IFM is required to run this notebook.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Add scripts/ to path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'scripts'))

from config import load_config, OUTPUTS_DIR, RESULTS_PATH

cfg = load_config()
print('Config loaded:', cfg.slice_T)

In [ ]:
# Load NPZ snapshots
npz_path = RESULTS_PATH.with_suffix('.npz')
data = np.load(str(npz_path))

times = data['times']   # float64[20]  — days
T     = data['T']       # float32[20, 28236]  — °C
h     = data['h']       # float32[20, 28236]  — m

print(f'Snapshots: {len(times)}')
print(f'Time range: {times[0]:.0f} d ... {times[-1]:.0f} d')
print(f'T shape: {T.shape}   T range: [{T.min():.1f}, {T.max():.1f}] °C')
print(f'h shape: {h.shape}   h range: [{h.min():.1f}, {h.max():.1f}] m')

In [ ]:
# Global mean temperature per snapshot
T_mean = T.mean(axis=1)
years  = times / 365.25

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(years, T_mean, 'o-')
ax.set_xlabel('Time [yr]')
ax.set_ylabel('Domain-mean temperature [°C]')
ax.set_title('Group 3 — domain-mean T over 100 years')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Temperature histogram at t=0 (ICs) vs t=100yr
nps = T.shape[1] // 6   # approx nodes per slice

# Use Slice 2 (top of reservoir): nodes [nps, 2*nps]
T0_s2  = cfg.slice_T[1]           # scalar — undisturbed IC
T100_s2 = T[-1, nps:2*nps]        # final snapshot, Slice 2

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(T100_s2, bins=60, alpha=0.7, label='t = 100 yr (Slice 2)')
ax.axvline(T0_s2, color='red', linestyle='--', label=f'IC = {T0_s2:.1f} °C')
ax.set_xlabel('Temperature [°C]')
ax.set_ylabel('Node count')
ax.set_title('Slice 2 temperature distribution at t = 100 yr')
ax.legend()
plt.tight_layout()
plt.show()